# Bab 10: Model Grafis

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab10/jaringan_bayes.py": "import itertools\nimport numpy as np\n\n# Jaringan Bayes diagnosis mesin press (peluang ilustratif)  #<1>\n# P: pelumasan kurang, L: beban berlebih, B: bantalan aus,\n# G: getaran tinggi, S: suhu tinggi, A: bunyi abnormal\ninduk = {\"P\": [], \"L\": [], \"B\": [\"P\", \"L\"], \"G\": [\"B\"], \"S\": [\"B\", \"L\"], \"A\": [\"G\"]}\ntabel = {  # p(variabel = 1 | nilai induk)\n    \"P\": {(): 0.10}, \"L\": {(): 0.20},\n    \"B\": {(0, 0): 0.02, (0, 1): 0.15, (1, 0): 0.30, (1, 1): 0.60},\n    \"G\": {(0,): 0.05, (1,): 0.80},\n    \"S\": {(0, 0): 0.05, (0, 1): 0.40, (1, 0): 0.50, (1, 1): 0.85},\n    \"A\": {(0,): 0.10, (1,): 0.70}}\nnama = list(induk)\n\ndef gabungan(x):                                   # aturan kali sepanjang graf  #<2>\n    p = 1.0\n    for v in nama:\n        p1 = tabel[v][tuple(x[u] for u in induk[v])]\n        p *= p1 if x[v] else 1 - p1\n    return p\n\ndef tanya(target, bukti):                          # inferensi dengan enumerasi  #<3>\n    num = den = 0.0\n    for nilai in itertools.product((0, 1), repeat=len(nama)):\n        x = dict(zip(nama, nilai))\n        if all(x[k] == v for k, v in bukti.items()):\n            p = gabungan(x); den += p\n            num += p * x[target]\n    return num / den\n\nprint(f\"p(B=1)                 = {tanya('B', {}):.3f}\")\nprint(f\"p(B=1 | G=1)           = {tanya('B', {'G': 1}):.3f}\")\nprint(f\"p(B=1 | G=1, S=1)      = {tanya('B', {'G': 1, 'S': 1}):.3f}\")\nprint(f\"p(B=1 | A=1)           = {tanya('B', {'A': 1}):.3f}\")\nprint(f\"p(P=1 | B=1)           = {tanya('P', {'B': 1}):.3f}\")      # explaining away  #<4>\nprint(f\"p(P=1 | B=1, L=1)      = {tanya('P', {'B': 1, 'L': 1}):.3f}\")\nprint(f\"p(P=1 | L=1)           = {tanya('P', {'L': 1}):.3f}   (P dan L saling bebas)\")\n",
"bab10/mrf_bersihkan.py": "import numpy as np\n\ndef citra_asli(h=48, w=64):                        # pola biner sederhana: +1 / -1\n    y, x = np.mgrid[0:h, 0:w]\n    z = -np.ones((h, w))\n    for cx, cy, r in [(16, 20, 9), (30, 16, 7), (27, 31, 8), (48, 26, 12)]:\n        z[(x - cx)**2 + (y - cy)**2 < r**2] = 1\n    z[38:44, 6:58] = 1\n    return z\n\nrng = np.random.default_rng(0)\nx_asli = citra_asli()\ny = np.where(rng.random(x_asli.shape) < 0.10, -x_asli, x_asli)   # 10% piksel dibalik  #<1>\n\ndef icm(y, beta=1.0, eta=2.1, putaran=10):         # iterated conditional modes  #<2>\n    x = y.copy()\n    for k in range(putaran):\n        berubah = 0\n        for i in range(x.shape[0]):\n            for j in range(x.shape[1]):\n                tetangga = sum(x[a, b] for a, b in ((i-1, j), (i+1, j), (i, j-1), (i, j+1))\n                               if 0 <= a < x.shape[0] and 0 <= b < x.shape[1])\n                baru = 1.0 if beta * tetangga + eta * y[i, j] > 0 else -1.0   #<3>\n                berubah += baru != x[i, j]; x[i, j] = baru\n        if berubah == 0:\n            break\n    return x, k + 1\n\ngalat = lambda z: np.mean(z != x_asli)\nprint(f\"citra berderau   : {galat(y):.2%} piksel salah\")\nfor beta in (0.0, 1.0, 3.0):\n    x, k = icm(y, beta=beta)\n    print(f\"ICM, beta = {beta:.1f} : {galat(x):.2%} piksel salah ({k} putaran)\")\n",
"bab10/sum_product_rantai.py": "import itertools, time\nimport numpy as np\n\n# Rantai N unit konveyor: x_n = 1 bila unit n bermasalah (ilustrasi).\n# Unit bertetangga cenderung sama kondisinya; sensor murah memberi bukti lemah.\nrng = np.random.default_rng(0)\nN = 16\npsi = np.array([[0.9, 0.1], [0.1, 0.9]])            # potensial pasangan  #<1>\nphi = np.c_[rng.uniform(0.5, 0.9, N), rng.uniform(0.1, 0.5, N)]   # bukti sensor\nphi[[5, 7, 8], 1] *= 20                             # sensor unit 6, 8, 9 panas\n\ndef brute_force():                                  # 2^N konfigurasi\n    marg, terbaik = np.zeros((N, 2)), (0, None)\n    for x in itertools.product((0, 1), repeat=N):\n        p = np.prod(phi[np.arange(N), x]) * np.prod(psi[x[:-1], x[1:]])\n        marg[np.arange(N), x] += p\n        if p > terbaik[0]: terbaik = (p, x)\n    return marg / marg.sum(1, keepdims=True), np.array(terbaik[1])\n\ndef sum_product():                                  # pesan maju dan mundur  #<2>\n    maju, mundur = np.ones((N, 2)), np.ones((N, 2))\n    for n in range(1, N):\n        m = (maju[n-1] * phi[n-1]) @ psi; maju[n] = m / m.sum()\n    for n in range(N - 2, -1, -1):\n        m = psi @ (mundur[n+1] * phi[n+1]); mundur[n] = m / m.sum()\n    b = maju * phi * mundur\n    return b / b.sum(1, keepdims=True)\n\ndef max_sum():                                      # Viterbi dalam log  #<3>\n    lpsi, lphi = np.log(psi), np.log(phi)\n    skor, asal = lphi[0].copy(), np.zeros((N, 2), int)\n    for n in range(1, N):\n        cal = skor[:, None] + lpsi\n        asal[n] = cal.argmax(0); skor = cal.max(0) + lphi[n]\n    x = [int(skor.argmax())]\n    for n in range(N - 1, 0, -1): x.append(asal[n, x[-1]])\n    return np.array(x[::-1])\n\nt0 = time.perf_counter(); m_bf, x_bf = brute_force(); t_bf = time.perf_counter() - t0\nt0 = time.perf_counter(); m_sp = sum_product(); x_ms = max_sum(); t_sp = time.perf_counter() - t0\nprint(\"p(bermasalah):\", \" \".join(f\"{v:.2f}\" for v in m_sp[:8, 1]))\nprint(\"              \", \" \".join(f\"{v:.2f}\" for v in m_sp[8:, 1]))\nprint(f\"selisih maks. marginal brute force vs sum-product: {np.abs(m_bf - m_sp).max():.1e}\")\nprint(\"MAP max-sum :\", \"\".join(map(str, x_ms)), \"| sama dengan brute force:\", np.array_equal(x_bf, x_ms))\nprint(f\"waktu: brute force {t_bf:.2f} s, pengiriman pesan {t_sp*1e3:.2f} ms\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab10')
print('folder kerja:', os.getcwd())

## Listing 10.1: Inferensi pada jaringan Bayes dengan enumerasi

Berkas: `kode/bab10/jaringan_bayes.py`

In [ ]:
import itertools
import numpy as np

# Jaringan Bayes diagnosis mesin press (peluang ilustratif)  # (1)
# P: pelumasan kurang, L: beban berlebih, B: bantalan aus,
# G: getaran tinggi, S: suhu tinggi, A: bunyi abnormal
induk = {"P": [], "L": [], "B": ["P", "L"], "G": ["B"], "S": ["B", "L"], "A": ["G"]}
tabel = {  # p(variabel = 1 | nilai induk)
    "P": {(): 0.10}, "L": {(): 0.20},
    "B": {(0, 0): 0.02, (0, 1): 0.15, (1, 0): 0.30, (1, 1): 0.60},
    "G": {(0,): 0.05, (1,): 0.80},
    "S": {(0, 0): 0.05, (0, 1): 0.40, (1, 0): 0.50, (1, 1): 0.85},
    "A": {(0,): 0.10, (1,): 0.70}}
nama = list(induk)

def gabungan(x):                                   # aturan kali sepanjang graf  # (2)
    p = 1.0
    for v in nama:
        p1 = tabel[v][tuple(x[u] for u in induk[v])]
        p *= p1 if x[v] else 1 - p1
    return p

def tanya(target, bukti):                          # inferensi dengan enumerasi  # (3)
    num = den = 0.0
    for nilai in itertools.product((0, 1), repeat=len(nama)):
        x = dict(zip(nama, nilai))
        if all(x[k] == v for k, v in bukti.items()):
            p = gabungan(x); den += p
            num += p * x[target]
    return num / den

print(f"p(B=1)                 = {tanya('B', {}):.3f}")
print(f"p(B=1 | G=1)           = {tanya('B', {'G': 1}):.3f}")
print(f"p(B=1 | G=1, S=1)      = {tanya('B', {'G': 1, 'S': 1}):.3f}")
print(f"p(B=1 | A=1)           = {tanya('B', {'A': 1}):.3f}")
print(f"p(P=1 | B=1)           = {tanya('P', {'B': 1}):.3f}")      # explaining away  # (4)
print(f"p(P=1 | B=1, L=1)      = {tanya('P', {'B': 1, 'L': 1}):.3f}")
print(f"p(P=1 | L=1)           = {tanya('P', {'L': 1}):.3f}   (P dan L saling bebas)")

## Listing 10.2: Membersihkan citra biner dengan MRF dan ICM

Berkas: `kode/bab10/mrf_bersihkan.py`

In [ ]:
import numpy as np

def citra_asli(h=48, w=64):                        # pola biner sederhana: +1 / -1
    y, x = np.mgrid[0:h, 0:w]
    z = -np.ones((h, w))
    for cx, cy, r in [(16, 20, 9), (30, 16, 7), (27, 31, 8), (48, 26, 12)]:
        z[(x - cx)**2 + (y - cy)**2 < r**2] = 1
    z[38:44, 6:58] = 1
    return z

rng = np.random.default_rng(0)
x_asli = citra_asli()
y = np.where(rng.random(x_asli.shape) < 0.10, -x_asli, x_asli)   # 10% piksel dibalik  # (1)

def icm(y, beta=1.0, eta=2.1, putaran=10):         # iterated conditional modes  # (2)
    x = y.copy()
    for k in range(putaran):
        berubah = 0
        for i in range(x.shape[0]):
            for j in range(x.shape[1]):
                tetangga = sum(x[a, b] for a, b in ((i-1, j), (i+1, j), (i, j-1), (i, j+1))
                               if 0 <= a < x.shape[0] and 0 <= b < x.shape[1])
                baru = 1.0 if beta * tetangga + eta * y[i, j] > 0 else -1.0   # (3)
                berubah += baru != x[i, j]; x[i, j] = baru
        if berubah == 0:
            break
    return x, k + 1

galat = lambda z: np.mean(z != x_asli)
print(f"citra berderau   : {galat(y):.2%} piksel salah")
for beta in (0.0, 1.0, 3.0):
    x, k = icm(y, beta=beta)
    print(f"ICM, beta = {beta:.1f} : {galat(x):.2%} piksel salah ({k} putaran)")

## Listing 10.3: Sum-product dan max-sum pada rantai 16 unit, dibanding enumerasi

Berkas: `kode/bab10/sum_product_rantai.py`

In [ ]:
import itertools, time
import numpy as np

# Rantai N unit konveyor: x_n = 1 bila unit n bermasalah (ilustrasi).
# Unit bertetangga cenderung sama kondisinya; sensor murah memberi bukti lemah.
rng = np.random.default_rng(0)
N = 16
psi = np.array([[0.9, 0.1], [0.1, 0.9]])            # potensial pasangan  # (1)
phi = np.c_[rng.uniform(0.5, 0.9, N), rng.uniform(0.1, 0.5, N)]   # bukti sensor
phi[[5, 7, 8], 1] *= 20                             # sensor unit 6, 8, 9 panas

def brute_force():                                  # 2^N konfigurasi
    marg, terbaik = np.zeros((N, 2)), (0, None)
    for x in itertools.product((0, 1), repeat=N):
        p = np.prod(phi[np.arange(N), x]) * np.prod(psi[x[:-1], x[1:]])
        marg[np.arange(N), x] += p
        if p > terbaik[0]: terbaik = (p, x)
    return marg / marg.sum(1, keepdims=True), np.array(terbaik[1])

def sum_product():                                  # pesan maju dan mundur  # (2)
    maju, mundur = np.ones((N, 2)), np.ones((N, 2))
    for n in range(1, N):
        m = (maju[n-1] * phi[n-1]) @ psi; maju[n] = m / m.sum()
    for n in range(N - 2, -1, -1):
        m = psi @ (mundur[n+1] * phi[n+1]); mundur[n] = m / m.sum()
    b = maju * phi * mundur
    return b / b.sum(1, keepdims=True)

def max_sum():                                      # Viterbi dalam log  # (3)
    lpsi, lphi = np.log(psi), np.log(phi)
    skor, asal = lphi[0].copy(), np.zeros((N, 2), int)
    for n in range(1, N):
        cal = skor[:, None] + lpsi
        asal[n] = cal.argmax(0); skor = cal.max(0) + lphi[n]
    x = [int(skor.argmax())]
    for n in range(N - 1, 0, -1): x.append(asal[n, x[-1]])
    return np.array(x[::-1])

t0 = time.perf_counter(); m_bf, x_bf = brute_force(); t_bf = time.perf_counter() - t0
t0 = time.perf_counter(); m_sp = sum_product(); x_ms = max_sum(); t_sp = time.perf_counter() - t0
print("p(bermasalah):", " ".join(f"{v:.2f}" for v in m_sp[:8, 1]))
print("              ", " ".join(f"{v:.2f}" for v in m_sp[8:, 1]))
print(f"selisih maks. marginal brute force vs sum-product: {np.abs(m_bf - m_sp).max():.1e}")
print("MAP max-sum :", "".join(map(str, x_ms)), "| sama dengan brute force:", np.array_equal(x_bf, x_ms))
print(f"waktu: brute force {t_bf:.2f} s, pengiriman pesan {t_sp*1e3:.2f} ms")